# Salary Prediction

## Table of Contents
- [Introduction](#intro)
- [Part I - Descriptive Statistics](#descriptive)
- [Part II - Regression](#regression)
- [Part III - Interpret Results](#interpret)


<a id='intro'></a>
### Introduction

Linear Regression is very commonly performed by data analysts and data scientists.  For this project, you will be working to understand the results of a Linear Regression model associated with salaries.  Your goal is to work through this notebook to understand what variables are related to salary, and how exactly they are related.

As a final check, assure you meet all the criteria on the rubric.

<a id='descriptive'></a>
#### Part I - Descriptive Statistics

To get started, let's import our libraries.

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

random.seed(0)

For each of the parts of question `1` notice links to [pandas documentation](https://pandas.pydata.org/) is provided to assist with answering the questions.  Though there are other ways you could solve the questions, the documentation is provided to assist you with one fast way to find the answer to each question.


`1.a)` Now, read in the `salary_data.csv` data. Store it in `df`. Read in the dataset and take a look at the top few rows here. **This question is completed for you**:

In [ ]:
df = pd.read_csv('salary_data.csv')
df.head()

`b)` Use the below cell to find the number of rows in the dataset. [Helpful pandas link - `Dataframe.shape`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html#pandas.DataFrame.shape)

In [ ]:
len(df)

`c)` Do any of the rows have missing values? [Helpful pandas link - `Dataframe.isnull`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html) and [helpful pandas link - `Dataframe.sum`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html)

If there are missing values, determine a method for dealing with them.

In [ ]:
df.isnull().sum()

In [ ]:
df[df.isnull().any(axis=1)]

In [ ]:
# Just remove the two rows since they don't have any data.
df = df.dropna()

df.isnull().sum()

`d)` How many employees are in each `Education Level`? Build a bar chart to show the count of employees in each level.

In [ ]:
df["Education Level"].value_counts()

In [ ]:
# bar chart of results - this part is done for you
df['Education Level'].value_counts().plot(kind='bar');
plt.title('Number of Employees From Each Education Level');
plt.ylabel('Count of Employees');
plt.show();

`e)` What are the possible values for `Salary`?  What does the distribution of `Salary` look like?

In [ ]:
# There is also a row that has a salary of 350, so this is probably a data entry error. I would be best to remove it so our results aren't skewed since we can't know for sure what this value should be.
df = df[df["Salary"] > 1000]

df["Salary"].value_counts()

# The salary appears right skewed.

In [ ]:
df["Salary"].hist()
plt.xlabel("Salary")
plt.ylabel("Frequency")
plt.title("Distribution of Salary")
plt.show()

<a id='regression'></a>
#### Part II - Regression

`1.` Now that you have had a chance to learn more about the dataset, let's look more at how different factors are related to `Salary`.

`a)` Consider average salary by gender, is there evidence that salaries are higher for one gender over the other? **This question is completed for you**

In [ ]:
df.groupby("Gender").mean(numeric_only=True)

`b)` Consider average salary by education level, is there evidence that salaries are higher for increased education?

In [ ]:
df.groupby("Education Level").mean(numeric_only=True)

# Yes salary does appear to increase with education level

`c)` Consider average salary by years of experience, is there evidence that salaries are associated with increased years of experience?

In [ ]:
df.groupby("Years of Experience").mean(numeric_only=True)

# Yes salary does appear to increase with years of experience

`d)`  To make use of Job Title column, let's create a bool flag based on word existence

List of words: 

* Director
* Junior
* Senior
* Manager
* Analyst
* Engineer

**This question is completed for you**

In [ ]:
flag_words = ['director', 'junior', 'senior', 'manager', 'analyst', 'engineer']
df['Job Title'] = df['Job Title'].str.lower()

for word in flag_words:
    df['is_' + word] = df['Job Title'].str.contains(word).astype(int)
    
df = df.drop('Job Title', axis=1)

`e)` Create a flag for gender where 1 is if a person is male and 0 if the person is not.

In [ ]:
df["Male"] = pd.get_dummies(df["Gender"], dtype=int)["Male"]

df = df.drop(columns="Gender")

df.head()

`f)` Use statsmodels to fit a linear model to predict salary using each of the features from `a-e`.  These include:
* Gender
* Job TItle
* Years of Experience
* Education

In [ ]:
df[["bachelors", "masters", "phd"]] = pd.get_dummies(df["Education Level"], dtype=int)

df = df.drop(columns=["Education Level", "bachelors"])

df.head()

In [ ]:
import statsmodels.api as sm

df["intercept"] = 1

lm = sm.OLS(df["Salary"], df[["intercept", "Years of Experience", "Male", "is_director", "is_junior", "is_senior", "is_manager", "is_analyst", "is_engineer", "masters", "phd",]])

results = lm.fit()

results.summary()

<a id='interpretation'></a>
### Part III - Interpret Results

`1.` Consider you are tasked with finding which features in your linear model are significantly related to salary.  Were there any features that were not significantly related to salary in your first model?  If not, remove those features and fit a new model.  Only keep the features that were significant from the original model.

In [ ]:
# some of the jobs titles such as is_junior, is_analyst, is_engineer aren't statistically significant

lm2 = sm.OLS(df["Salary"], df[["intercept", "Years of Experience", "Male", "is_director", "is_senior", "is_manager", "masters", "phd",]])

results2 = lm2.fit()

results2.summary()

`a)` With each additional year of experience, what is the expected change in salary?  What is the 95% confidence interval of the change?

**5,081.26 with a 95% confidence interval of 4,736.53 to 5,425.98**

`b)` What is the expected difference in salary between someone with a senior title and someone with none of the other title indications?

**14,350**

`c)` What is the expected difference in salary between someone with a PhD and an individual with no PhD nor master's degree?  What is the 95% confidence interval of the change?

**23,140 with a 95% confidence interval of 17,800 to 28,500**

`d)` If a male employee has 5 years of experience as a senior engineer with a bachelor's degree, what is the expected salary of the employee?

**29,070 + 7,848.84 + 14,350 + 5 * 5,081.26 = 76,675.14**

Note: We dropped the engineer flag from this model so it wasn't included in the salary calculation

`e)` Imagine that the employee in question `d)` actually has a salary of $110,000, what would the residual be for this employee?

**110,000 - 76,675.14 = 33,324.86**

`f)` How well do you think your model fits?  What metrics or plots would you consider to understand if this model does a good job of predicting salary?

In [ ]:
# Actual vs Predicted
plt.scatter(results2.fittedvalues, df["Salary"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.plot([0, 200000], [0, 200000], color='r')
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
results2.resid.hist(bins=30)
plt.xlabel("Residual")
plt.title("Distribution of Residuals")
plt.show()

R-squared (0.914) — 91.4% of the variance in salary is explained by the model. Generally strong.

**Based on the plots, R-squared and p-values the model appears to fit the data well**